In [1]:
# ============================================================
# 🎯 META MMS LANGUAGE IDENTIFICATION + VAD
# Model: facebook/mms-lid-4017 (4,017 languages including Kabyle)
# ============================================================

!pip install transformers torch librosa webrtcvad numpy --quiet

import torch
import librosa
import numpy as np

from transformers import Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
import webrtcvad
import struct

print("📦 Installing and loading Meta MMS-LID-4017...")
print("   (This model supports 4,017 languages including Kabyle 'kab')")

# Load the MMS Language Identification model
model_id = "facebook/mms-lid-4017"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"✅ MMS-LID-4017 loaded on {device.upper()}")
print(f"   Total languages supported: {model.config.num_labels}")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


📦 Installing and loading Meta MMS-LID-4017...
   (This model supports 4,017 languages including Kabyle 'kab')


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 22.28 GiB of which 7.12 MiB is free. Process 33056 has 20.00 GiB memory in use. Process 118840 has 2.26 GiB memory in use. Of the allocated memory 2.03 GiB is allocated by PyTorch, and 47.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
# ============================================================
# 🔊 VOICE ACTIVITY DETECTION (VAD) + LANGUAGE IDENTIFICATION
# Segments audio into speech chunks, then identifies language
# ============================================================

def frame_generator(audio, sample_rate, frame_duration_ms=30):
    """Generate audio frames for VAD processing"""
    n = int(sample_rate * (frame_duration_ms / 1000.0))
    offset = 0
    while offset + n <= len(audio):
        yield audio[offset:offset + n]
        offset += n

def vad_collector(audio, sample_rate, aggressiveness=2, frame_duration_ms=30, padding_duration_ms=300):
    """
    Collect speech segments using WebRTC VAD
    aggressiveness: 0-3 (3 = most aggressive filtering of non-speech)
    """
    vad = webrtcvad.Vad(aggressiveness)
    
    # Convert to 16-bit PCM for VAD
    audio_int16 = (audio * 32767).astype(np.int16)
    
    frames = list(frame_generator(audio_int16, sample_rate, frame_duration_ms))
    
    num_padding_frames = int(padding_duration_ms / frame_duration_ms)
    ring_buffer = []
    triggered = False
    
    voiced_frames = []
    segments = []
    segment_start = 0
    
    for i, frame in enumerate(frames):
        # Convert frame to bytes for VAD
        frame_bytes = struct.pack(f"{len(frame)}h", *frame)
        
        is_speech = vad.is_speech(frame_bytes, sample_rate)
        
        if not triggered:
            ring_buffer.append((i, frame, is_speech))
            num_voiced = len([f for _, f, speech in ring_buffer if speech])
            
            if num_voiced > 0.9 * len(ring_buffer):
                triggered = True
                segment_start = ring_buffer[0][0] * int(sample_rate * frame_duration_ms / 1000)
                voiced_frames = [f for _, f, _ in ring_buffer]
                ring_buffer = []
        else:
            voiced_frames.append(frame)
            ring_buffer.append((i, frame, is_speech))
            num_unvoiced = len([f for _, f, speech in ring_buffer if not speech])
            
            if num_unvoiced > 0.9 * len(ring_buffer):
                triggered = False
                # Save segment
                segment_audio = np.concatenate(voiced_frames)
                segment_end = segment_start + len(segment_audio)
                segments.append({
                    'start_sample': segment_start,
                    'end_sample': segment_end,
                    'audio': segment_audio.astype(np.float32) / 32767.0  # Back to float
                })
                ring_buffer = []
                voiced_frames = []
        
        if len(ring_buffer) > num_padding_frames:
            ring_buffer.pop(0)
    
    # Handle remaining audio
    if voiced_frames:
        segment_audio = np.concatenate(voiced_frames)
        segment_end = segment_start + len(segment_audio)
        segments.append({
            'start_sample': segment_start,
            'end_sample': segment_end,
            'audio': segment_audio.astype(np.float32) / 32767.0
        })
    
    return segments

def identify_language(audio_segment, sample_rate=16000, top_k=5):
    """
    Identify language using MMS-LID-4017
    Returns top-k language predictions with probabilities
    """
    with torch.no_grad():
        inputs = feature_extractor(audio_segment, sampling_rate=sample_rate, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        
        # Get top-k predictions
        top_probs, top_indices = torch.topk(probs[0], top_k)
        
        results = []
        for prob, idx in zip(top_probs, top_indices):
            lang_code = model.config.id2label[idx.item()]
            results.append({
                'language': lang_code,
                'probability': prob.item()
            })
        
        return results

print("✅ VAD + Language Identification functions ready!")
print("\n📋 Functions available:")
print("   - vad_collector(): Segments audio into speech chunks")
print("   - identify_language(): Identifies language of audio segment")

📦 Loading MMS-LID-4017 for fine-tuning...


✅ Found Kabyle label: 'kab' → ID 780

📊 Dataset ready:
   Train samples: 46641
   Validation samples: 5183


In [4]:
# ============================================================
# 🚀 FINE-TUNING CONFIGURATION
# ============================================================

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finetune_model = finetune_model.to(device)

print(f"🖥️ Training on: {device}")

# Freeze most of the model, only train classifier head
# This is more efficient and prevents catastrophic forgetting
print("\n🔒 Freezing base model layers...")

# Freeze all wav2vec2 layers
for param in finetune_model.wav2vec2.parameters():
    param.requires_grad = False

# Only train the classifier (projection + classifier layers)
trainable_params = 0
total_params = 0
for name, param in finetune_model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"   ✅ Training: {name}")

print(f"\n📊 Parameters:")
print(f"   Total: {total_params:,}")
print(f"   Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

# ============================================================
# 📈 CUSTOM TRAINING LOOP
# ============================================================

def collate_fn(batch):
    """Custom collate function to handle variable length audio"""
    input_values = torch.stack([item['input_values'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    return {'input_values': input_values, 'labels': labels}

# Create data loaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=8,  # Adjust based on GPU memory
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print(f"\n📦 DataLoaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")

🖥️ Training on: cuda

🔒 Freezing base model layers...
   ✅ Training: projector.weight
   ✅ Training: projector.bias
   ✅ Training: classifier.weight
   ✅ Training: classifier.bias

📊 Parameters:
   Total: 970,077,745
   Trainable: 5,429,169 (0.56%)

📦 DataLoaders created:
   Train batches: 5831
   Val batches: 648


In [ ]:
# ============================================================
# 🎯 FINE-TUNING TRAINING LOOP
# ============================================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

# Training hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

# Optimizer (only for trainable params)
optimizer = AdamW(
    filter(lambda p: p.requires_grad, finetune_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * len(train_loader))

# Loss function (CrossEntropy for classification)
criterion = nn.CrossEntropyLoss()

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': [],
    'val_kabyle_recall': []
}

print("="*70)
print("🚀 STARTING FINE-TUNING")
print("="*70)
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Batch Size: 8")
print(f"   Target: Improve Kabyle ('kab') detection")
print("="*70)

best_val_accuracy = 0.0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    print(f"\n📍 Epoch {epoch + 1}/{NUM_EPOCHS}")
    
    # ========== TRAINING ==========
    finetune_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for batch in pbar:
        input_values = batch['input_values'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = finetune_model(input_values)
        loss = criterion(outputs.logits, labels)
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        
        # Calculate accuracy
        predictions = torch.argmax(outputs.logits, dim=-1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100*train_correct/train_total:.1f}%'
        })
    
    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total
    
    # ========== VALIDATION ==========
    finetune_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    kabyle_correct = 0
    kabyle_total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_values = batch['input_values'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = finetune_model(input_values)
            loss = criterion(outputs.logits, labels)
            
            val_loss += loss.item()
            
            predictions = torch.argmax(outputs.logits, dim=-1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)
            
            # Kabyle-specific recall
            kabyle_mask = (labels == kabyle_label_id)
            kabyle_correct += ((predictions == labels) & kabyle_mask).sum().item()
            kabyle_total += kabyle_mask.sum().item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total
    kabyle_recall = 100 * kabyle_correct / kabyle_total if kabyle_total > 0 else 0
    
    # Save history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(val_accuracy)
    history['val_kabyle_recall'].append(kabyle_recall)
    
    print(f"\n   📊 Results:")
    print(f"      Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.1f}%")
    print(f"      Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.1f}%")
    print(f"      🎯 Kabyle Recall: {kabyle_recall:.1f}%")
    
    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = finetune_model.state_dict().copy()
        print(f"      ✅ New best model saved! (Accuracy: {val_accuracy:.1f}%)")

print("\n" + "="*70)
print("✅ FINE-TUNING COMPLETE!")
print("="*70)
print(f"   Best Validation Accuracy: {best_val_accuracy:.1f}%")

🚀 STARTING FINE-TUNING
   Epochs: 5
   Learning Rate: 0.0001
   Batch Size: 8
   Target: Improve Kabyle ('kab') detection

📍 Epoch 1/5


Training:   9%|▉         | 514/5831 [21:15<3:35:58,  2.44s/it, loss=0.0003, acc=98.5%]